# Session 3.2 -- Augmented Retrieval & the Embedding Gap

**AI-2: AI Backend Engineering**

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Build** a filter classification function that maps question intent to ChromaDB `where` clauses
2. **Implement** a comparison scoring function that measures naive vs filtered retrieval quality
3. **Wire** an auto-classify-and-retrieve pipeline that routes queries to the right retrieval strategy
4. **Observe** (via PCA visualization) that questions and answers cluster in different embedding neighborhoods -- the "embedding gap"

## What You Are Building Today

| Task | Scaffold | What You Write |
|------|----------|----------------|
| `build_filter()` | YELLOW | Classify question intent → return ChromaDB `where` clause |
| `score_comparison()` | YELLOW | Compare naive vs filtered results → compute precision, overlap, verdict |
| `auto_classify_and_retrieve()` | ORANGE | Wire everything together: classify → filter → retrieve → score → explain |

## Where We Are in the Pipeline

| Session | What We Built | Status |
|---------|--------------|--------|
| **1.1** | API connection + generation | Done |
| **1.2** | Structured extraction with Pydantic | Done |
| **2.1** | Embeddings + similarity measurement | Done |
| **2.2** | Chunking + vector store ingestion | Done |
| **3.1** | Naive RAG -- retrieve + generate | Done |
| **3.2** | Metadata-filtered RAG + comparison | **Today** |
| **4.1** | Observability and debugging | Next session |

---

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Find repo root by walking up until we find pyproject.toml
_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

from dotenv import load_dotenv
load_dotenv()

In [ ]:
import time
import textwrap
import numpy as np

# Naive RAG from Session 3.1
from src.s4_retrieval.naive import naive_retrieve, naive_rag, RAGResponse

# Filtered RAG -- provided complete for this session
from src.s4_retrieval.filtered import filtered_retrieve, filtered_rag, compare_retrieval

# Generation module
from src.s0_generation.generate import call_claude_with_usage

# Embedding module
from src.s2_embeddings.embed import embed_texts, cosine_similarity

# ChromaDB collection
from src.s3_ingestion.store import get_collection

print("All imports loaded successfully.")

In [ ]:
# Verify the ChromaDB collection is populated AND has doc_type metadata
# If doc_type is missing, re-ingest with classification
collection = get_collection()
print(f"Collection: {collection.name}")
print(f"Total chunks stored: {collection.count()}")

if collection.count() == 0:
    print("\nWARNING: Collection is empty. Run session_2_2.ipynb first to ingest the corpus.")
else:
    # Check if doc_type metadata is populated
    all_meta = collection.get(include=["metadatas"], limit=collection.count())
    doc_types_found = set(m.get("doc_type", "unknown") for m in all_meta["metadatas"])

    if doc_types_found <= {"unknown", None}:
        print("\ndoc_type metadata is missing -- re-ingesting with classification...")
        print("(This can happen if Session 2.2 ingestion did not include doc_type)\n")

        from src.s3_ingestion.chunker import chunk_document
        from src.s3_ingestion.store import ingest_chunks, reset_collection

        data_dir = _root / "data" / "northbrook"
        if not data_dir.exists():
            data_dir = _root / "AI-2" / "data" / "northbrook"

        def classify_doc_type(filename):
            name = filename.lower()
            if "memo" in name:
                return "memo"
            elif "policy" in name or "handbook" in name:
                return "policy"
            elif "financial" in name:
                return "financial"
            elif "meeting" in name or "standup" in name or "committee" in name:
                return "meeting"
            return "unknown"

        reset_collection()
        all_files = sorted(data_dir.glob("*.md"))
        total = 0
        for f in all_files:
            doc_type = classify_doc_type(f.name)
            chunks = chunk_document(f.read_text(), source=f.name, doc_type=doc_type)
            ingest_chunks(chunks)
            total += len(chunks)
            print(f"  {f.name:<40} type={doc_type:<10} chunks={len(chunks)}")

        collection = get_collection()
        print(f"\nRe-ingested {total} chunks with doc_type metadata.")
        doc_types_found = set(m.get("doc_type") for m in collection.get(include=["metadatas"], limit=collection.count())["metadatas"])

    print(f"\ndoc_type values: {sorted(doc_types_found)}")
    print("Collection is ready for filtered retrieval.")

---

## 2. Recap: Where Naive RAG Broke

In Session 3.1, you built naive RAG and then you broke it. You found semantic drift, hallucinations, and wrong document types in your results. Let's re-run a few of those failure cases to remind ourselves what went wrong.

The key insight from 3.1: **semantic similarity finds things that are ABOUT the same topic, but "about the same topic" and "useful for answering this specific question" are not the same thing.**

In [ ]:
# Failure Case 1: Wrong document type in results
# Asking about vacation policy -- does naive retrieval stay focused?

query_1 = "What is Northbrook Partners' vacation policy?"
naive_results_1 = naive_retrieve(query_1, top_k=5)

print(f"Query: {query_1}")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in naive_results_1:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

# Count how many different doc_types appear
doc_types = set(r["metadata"].get("doc_type", "?") for r in naive_results_1)
print(f"\nDocument types in results: {doc_types}")
if len(doc_types) == 1:
    print("Naive got lucky here -- all results are the right type.")
    print("But will it always be this clean? Check the next two cases.")
else:
    print("Notice: naive retrieval pulls from multiple document types, not just policies.")

In [ ]:
# Failure Case 2: Semantic drift -- topic-adjacent but not useful
# A question about a specific meeting decision pulls in random related content

query_2 = "What decisions were made about remote work at the board meeting?"
naive_results_2 = naive_retrieve(query_2, top_k=5)

print(f"Query: {query_2}")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in naive_results_2:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

print("\nDoes every result actually contain a DECISION about remote work?")
print("Or did some just mention 'remote work' in passing?")

In [ ]:
# Failure Case 3: Financial question pulling non-financial sources

query_3 = "What were the Q3 revenue numbers?"
naive_results_3 = naive_retrieve(query_3, top_k=5)

print(f"Query: {query_3}")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in naive_results_3:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

doc_types = set(r["metadata"].get("doc_type", "?") for r in naive_results_3)
print(f"\nDocument types in results: {doc_types}")
print("Financial data should come from financial documents, not meeting notes.")

**The pattern across these failure cases:** Naive retrieval returns whatever is semantically closest, regardless of document type. Sometimes it gets lucky (the vacation policy query may return all-policy results). But for cross-topic queries like "board meeting decisions about remote work" or "Q3 revenue numbers," it pulls from whatever documents mention those terms -- meeting notes, memos, financial reports all mixed together.

**The inconsistency is the problem.** You cannot predict when naive will work and when it will fail. Today's question: what if we could tell the retrieval system -- *"only look at policy documents"* or *"only look at financial reports"*? That is metadata filtering.

---

## 3. Introducing Metadata Filters

ChromaDB supports metadata filtering at query time via the `where` parameter. This is one line of code -- but it fundamentally changes retrieval behavior.

**Key concept:** Filters run BEFORE similarity search. ChromaDB first narrows the candidate set by metadata, then ranks by similarity within that subset. This is not post-filtering -- you are not retrieving 100 chunks and throwing away 80.

### What metadata is stored on our chunks?

Let's inspect the actual metadata fields available in our ChromaDB collection.

In [ ]:
# Peek at the metadata fields in our collection
collection = get_collection()
sample = collection.peek(limit=10)

print(f"Sample of {len(sample['ids'])} chunks from the collection:\n")

# Show all metadata keys and values
all_keys = set()
for meta in sample["metadatas"]:
    all_keys.update(meta.keys())

print(f"Metadata fields available: {sorted(all_keys)}\n")
print("Sample metadata for each chunk:")
print("-" * 60)
for i, meta in enumerate(sample["metadatas"]):
    print(f"  Chunk {i}: {meta}")

In [ ]:
# What distinct values exist for each metadata field?
# This tells us what filters we can actually build.

# Get a larger sample to see all distinct values
all_data = collection.get(include=["metadatas"], limit=collection.count())

print("Distinct values for each metadata field:\n")
for key in sorted(all_keys):
    if key == "chunk_index":
        # Skip chunk_index -- too many values
        values = set(m.get(key) for m in all_data["metadatas"])
        print(f"  {key:15s}: {len(values)} unique values (0 through {max(values)})")
    else:
        values = sorted(set(str(m.get(key, "")) for m in all_data["metadatas"]))
        print(f"  {key:15s}: {values}")

print("\nYour filters can ONLY use fields that exist in your metadata.")
print("If you did not add 'department' during ingestion, you cannot filter on it now.")

### ChromaDB `where` Clause Syntax

```python
# Exact match
where={"doc_type": "policy"}

# Comparison operators
where={"year": {"$gte": 2024}}

# Logical AND -- both conditions must match
where={
    "$and": [
        {"doc_type": "policy"},
        {"department": "HR"}
    ]
}

# Logical OR -- either condition matches
where={
    "$or": [
        {"doc_type": "policy"},
        {"doc_type": "handbook"}
    ]
}
```

**Important:** Every `$and` condition makes the filter stricter. Two conditions might be fine. Four might give you zero results. Always test.

---

## 4. Exploring `filtered_retrieve()`

The pre-built module `src/s4_retrieval/filtered.py` provides `filtered_retrieve()`. It is almost identical to `naive_retrieve()` -- one extra parameter: `where=filters`. That is it. The entire difference between naive and filtered retrieval is one keyword argument.

Let's explore how different filters change retrieval behavior.

In [ ]:
# Basic filter: only policy documents
query = "What is Northbrook Partners' vacation policy?"

filtered_results = filtered_retrieve(
    query,
    filters={"doc_type": "policy"},
    top_k=5
)

print(f"Query: {query}")
print(f"Filter: doc_type = 'policy'")
print(f"\nResults: {len(filtered_results)} chunks")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in filtered_results:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

print("\nEvery result is a policy document. No meeting notes, no financial reports.")

In [ ]:
# Filter by specific source document
# Useful when you know exactly which document should contain the answer

filtered_by_source = filtered_retrieve(
    query,
    filters={"source": "vacation_policy_2025.md"},
    top_k=5
)

print(f"Query: {query}")
print(f"Filter: source = 'vacation_policy_2025.md'")
print(f"\nResults: {len(filtered_by_source)} chunks")
print(f"\n{'Score':>8}  {'Source':<45}  {'Chunk':>6}")
print("-" * 65)
for r in filtered_by_source:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('chunk_index', '?'):>6}")

print("\nAll results from a single document. Maximum precision.")

In [ ]:
# Filter by document type: meeting notes only
meeting_query = "What decisions were made about remote work at the board meeting?"

filtered_meetings = filtered_retrieve(
    meeting_query,
    filters={"doc_type": "meeting"},
    top_k=5
)

print(f"Query: {meeting_query}")
print(f"Filter: doc_type = 'meeting'")
print(f"\nResults: {len(filtered_meetings)} chunks")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in filtered_meetings:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

In [ ]:
# Using $or to combine document types
# Useful when the answer might be in either policies OR handbooks

filtered_or = filtered_retrieve(
    query,
    filters={
        "$or": [
            {"doc_type": "policy"},
            {"doc_type": "memo"}
        ]
    },
    top_k=5
)

print(f"Query: {query}")
print(f"Filter: doc_type IN ('policy', 'memo')")
print(f"\nResults: {len(filtered_or)} chunks")
print(f"\n{'Score':>8}  {'Source':<45}  {'Type':<12}")
print("-" * 75)
for r in filtered_or:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}  {r['metadata'].get('doc_type', '?'):<12}")

### The Over-Filtering Problem

More filters is not always better. Each filter removes candidates. At some point, you remove the right answer.

In [ ]:
# Over-filtering: contradictory conditions = zero results
# vacation_policy_2025.md is classified as "policy", not "financial"
# Asking for doc_type="financial" AND source="vacation_policy_2025.md" matches nothing

ceo_query = "What did the CEO say about company direction?"

over_filtered = filtered_retrieve(
    ceo_query,
    filters={
        "$and": [
            {"doc_type": "financial"},
            {"source": "vacation_policy_2025.md"}
        ]
    },
    top_k=5
)

print(f"Query: {ceo_query}")
print(f"Filter: doc_type='financial' AND source='vacation_policy_2025.md'")
print(f"\nResults: {len(over_filtered)} chunks")

if len(over_filtered) == 0:
    print("\nZero results. The filter conditions are contradictory:")
    print("  vacation_policy_2025.md is classified as 'policy', not 'financial'")
    print("  No document can satisfy both conditions.")
    print("\nOver-filtering is WORSE than no filtering at all.")
else:
    print("\nUnexpected: got results despite contradictory filter.")
    for r in over_filtered:
        print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} type={r['metadata'].get('doc_type', '?')}")

In [ ]:
# Compare: the same CEO query with the correct filter
# The CEO kickoff is in memo_ceo_annual_kickoff.md -- classified as "memo"

correctly_filtered = filtered_retrieve(
    ceo_query,
    filters={"doc_type": "memo"},
    top_k=5
)

print(f"Query: {ceo_query}")
print(f"Filter: doc_type = 'memo'")
print(f"\nResults: {len(correctly_filtered)} chunks")
print(f"\n{'Score':>8}  {'Source':<45}")
print("-" * 55)
for r in correctly_filtered:
    print(f"{r['score']:>8.3f}  {r['metadata'].get('source', '?'):<45}")

print("\nCorrect filter type = relevant results.")
print("The engineering question: how do you CHOOSE the right filter for a query?")

**Takeaway:** Filtering is a precision tool. Applied correctly, it removes noise. Applied incorrectly, it removes the answer.

- **Filtering helps when** the question implies a specific document type, time period, or category
- **Filtering hurts when** the filter is too restrictive, the answer spans multiple document types, or the metadata is inconsistent

**The precision/recall tradeoff:**
- *Naive RAG:* high recall (considers everything, unlikely to miss relevant chunks), lower precision (includes irrelevant chunks too)
- *Filtered RAG:* higher precision (everything matches the filter), risk of lower recall (might filter out relevant chunks)

---

## 5. Build Your Filter Logic

You have seen how metadata filters work. You have seen the over-filtering trap. Now **you** write the logic that decides which filter to apply for a given question.

This is the core skill: given a question, analyze its intent and return the right ChromaDB `where` clause. Get this wrong and you get zero results (over-filtering) or noisy results (under-filtering).

**Connection to Session 3.1:** Remember the vocabulary mismatch problem -- "WFH" missed remote_work_policy because the embedding model could not bridge that vocabulary gap. Filtering by `doc_type` does not fix vocabulary mismatch, but it DOES fix the wrong-document-type problem you saw in the recap.

In [ ]:
# Helper function to display side-by-side retrieval results
def show_comparison(query, filters, filter_description):
    """Run naive and filtered retrieval and display results side by side."""
    naive_results = naive_retrieve(query, top_k=5)
    filtered_results = filtered_retrieve(query, filters=filters, top_k=5)

    print(f"Query: {query}")
    print(f"Filter: {filter_description}")
    print()

    print("=== NAIVE RETRIEVAL ===")
    for r in naive_results:
        print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?'):<40} type={r['metadata'].get('doc_type', '?')}")

    print()
    print("=== FILTERED RETRIEVAL ===")
    if not filtered_results:
        print("  [No results -- filter too restrictive]")
    else:
        for r in filtered_results:
            print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?'):<40} type={r['metadata'].get('doc_type', '?')}")

    # Show overlap
    naive_texts = {r["text"][:100] for r in naive_results}
    filtered_texts = {r["text"][:100] for r in filtered_results}
    overlap = len(naive_texts & filtered_texts)
    print(f"\nOverlap: {overlap}/{min(len(naive_results), len(filtered_results))} chunks in common")
    print("-" * 75)

In [ ]:
def build_filter(question: str) -> dict | None:
    """Analyze a question and return the appropriate ChromaDB where clause.

    Examine the question text to determine what type of document would
    contain the answer. Return a ChromaDB where dict, or None if no
    filter applies (let naive retrieval handle it).

    Args:
        question: The user's question

    Returns:
        A ChromaDB where clause dict (e.g., {"doc_type": "policy"}),
        or None if naive retrieval should be used.
    """
    question_lower = question.lower()

    # Step 1: Check for financial indicators
    financial_keywords = ["revenue", "profit", "earnings", "financial", "q3", "q4", "quarterly"]
    if any(kw in question_lower for kw in financial_keywords):
        return {"doc_type": "financial"}

    # Step 2: Check for meeting indicators
    meeting_keywords = ["board meeting", "standup", "committee", "discussed", "decided", "engineering sync"]
    if any(kw in question_lower for kw in meeting_keywords):
        return {"doc_type": "meeting"}

    # Step 3: Check for memo indicators
    memo_keywords = ["ceo", "announcement", "relocation", "security update", "kickoff", "memo"]
    if any(kw in question_lower for kw in memo_keywords):
        return {"doc_type": "memo"}

    # Step 4: Check for policy indicators
    policy_keywords = ["policy", "handbook", "vacation", "expense", "remote work", "dress code", "pto", "reimbursement"]
    if any(kw in question_lower for kw in policy_keywords):
        return {"doc_type": "policy"}

    # Step 5: Default -- no clear type
    return None

In [ ]:
# Test your build_filter() on these queries
# Each should return the correct doc_type filter (or None)

test_cases = [
    ("What is Northbrook's vacation policy?", "policy"),
    ("What were the Q3 revenue numbers?", "financial"),
    ("What decisions were made at the board meeting?", "meeting"),
    ("What did the CEO say about company direction?", "memo"),
    ("What services does Northbrook offer?", None),  # ambiguous -- should return None
    ("What is the expense reimbursement limit?", "policy"),
    ("How is the cloud migration progressing?", None),  # no meeting keywords -- returns None
    ("When is the office relocation?", "memo"),
]

print(f"{'Question':<55} {'Expected':>10} {'Got':>10} {'Match':>6}")
print("-" * 85)
correct = 0
for question, expected in test_cases:
    result = build_filter(question)
    got = result.get("doc_type") if result else None
    match = got == expected
    correct += match
    print(f"{question:<55} {str(expected):>10} {str(got):>10} {'  Y' if match else '  -':>6}")

print(f"\nScore: {correct}/{len(test_cases)}")
if correct == len(test_cases):
    print("Perfect! Your filter logic handles all test cases.")
elif correct >= 6:
    print("Good -- most cases covered. Check the misses.")
else:
    print("Review your keyword checks -- some cases are not matching.")

In [ ]:
# Now use YOUR build_filter() to fix the 3 failure cases from the recap
# This is the same show_comparison() helper, but using your filter logic

failure_queries = [
    "What is Northbrook Partners' vacation policy?",
    "What decisions were made about remote work at the board meeting?",
    "What were the Q3 revenue numbers?",
]

for query in failure_queries:
    my_filter = build_filter(query)
    if my_filter is None:
        print(f"\nQuery: {query}")
        print(f"  build_filter returned None -- no filter applied")
        print(f"  This query would use naive retrieval.")
        print("-" * 75)
    else:
        show_comparison(
            query,
            filters=my_filter,
            filter_description=f"build_filter() -> {my_filter}"
        )
    print()

**Check your results above:**

1. Did your `build_filter()` pick the right filter for each failure case?
2. Are the filtered results all from the expected document type?
3. Compare the naive vs filtered sources -- which set would produce a better answer?

The filtered sources might have **lower similarity scores** but they are from the **right kind of document**. This is the key insight: score measures similarity, not usefulness.

**Next:** You have a function that picks filters. Now you need a function that **measures** whether filtering actually helped.

---

## 6. Score Your Comparisons

You can see by eye that filtered results are often better. But "I looked at it and it seems better" does not scale to 100 queries. You need **metrics**.

Build a function that takes naive and filtered results and computes:
- **Source precision:** what fraction of results come from the expected document type?
- **Score delta:** difference in top similarity scores
- **Overlap:** how many chunks appear in both result sets?
- **Verdict:** which pipeline won?

This function is the foundation for Lab 2's comparison report.

In [ ]:
def score_comparison(
    question: str,
    naive_sources: list[dict],
    filtered_sources: list[dict],
    expected_type: str,
) -> dict:
    """Compare naive vs filtered retrieval results with metrics.

    Args:
        question: The query that was run
        naive_sources: Results from naive_retrieve()
        filtered_sources: Results from filtered_retrieve()
        expected_type: The doc_type that SHOULD contain the answer

    Returns:
        dict with keys: "question", "naive_precision", "filtered_precision",
        "naive_top_score", "filtered_top_score", "overlap", "verdict"
    """
    # Step 1: Source precision for naive results
    naive_correct = sum(
        1 for r in naive_sources
        if r["metadata"].get("doc_type") == expected_type
    )
    naive_precision = naive_correct / len(naive_sources) if naive_sources else 0.0

    # Step 2: Source precision for filtered results
    filtered_correct = sum(
        1 for r in filtered_sources
        if r["metadata"].get("doc_type") == expected_type
    )
    filtered_precision = filtered_correct / len(filtered_sources) if filtered_sources else 0.0

    # Step 3: Top similarity scores
    naive_top = naive_sources[0]["score"] if naive_sources else 0.0
    filtered_top = filtered_sources[0]["score"] if filtered_sources else 0.0

    # Step 4: Overlap -- chunks appearing in both result sets
    naive_texts = {r["text"][:100] for r in naive_sources}
    filtered_texts = {r["text"][:100] for r in filtered_sources}
    overlap = len(naive_texts & filtered_texts)

    # Step 5: Verdict
    if filtered_precision > naive_precision:
        verdict = "filtered_better"
    elif naive_precision > filtered_precision:
        verdict = "naive_better"
    else:
        verdict = "tie"

    return {
        "question": question,
        "naive_precision": naive_precision,
        "filtered_precision": filtered_precision,
        "naive_top_score": naive_top,
        "filtered_top_score": filtered_top,
        "overlap": overlap,
        "verdict": verdict,
    }

In [ ]:
# Test score_comparison() on the 3 failure cases

test_queries = [
    ("What is Northbrook Partners' vacation policy?", "policy"),
    ("What decisions were made about remote work at the board meeting?", "meeting"),
    ("What were the Q3 revenue numbers?", "financial"),
]

print(f"{'Question':<55} {'N-Prec':>7} {'F-Prec':>7} {'Overlap':>8} {'Verdict':<16}")
print("-" * 100)

for question, expected_type in test_queries:
    naive_src = naive_retrieve(question, top_k=5)
    my_filter = build_filter(question)
    filtered_src = filtered_retrieve(question, filters=my_filter, top_k=5) if my_filter else naive_src

    result = score_comparison(question, naive_src, filtered_src, expected_type)

    print(
        f"{question:<55} "
        f"{result['naive_precision']:>6.0%} "
        f"{result['filtered_precision']:>6.0%} "
        f"{result['overlap']:>8} "
        f"{result['verdict']:<16}"
    )

print("\nFiltered precision should be higher than naive for these cases.")
print("If not, check your build_filter() -- is it returning the right doc_type?")

---

## 7. The Smart Pipeline

You have `build_filter()` to pick the right filter and `score_comparison()` to measure quality. Now wire them together into a single function that:

1. **Classifies** the question (using your `build_filter()`)
2. **Retrieves** using the right strategy (filtered or naive)
3. **Falls back** to naive if filtering returns zero results
4. **Explains** its reasoning

This is the pattern you will use in Lab 2. It is also a preview of the routing logic you will build in the AG certificate.

In [ ]:
def auto_classify_and_retrieve(question: str, top_k: int = 5) -> dict:
    """Classify a question, decide whether to filter, retrieve, and explain.

    This is the full pipeline: analyze the question, build a filter (or not),
    run retrieval, and return everything with reasoning.

    Args:
        question: The user's question
        top_k: Number of chunks to retrieve

    Returns:
        dict with keys: "question", "method", "filter_applied",
        "sources", "reasoning", "top_score", "source_types"
    """
    # Step 1: Classify the question
    my_filter = build_filter(question)

    # Step 2: Route to the appropriate retrieval strategy
    if my_filter is not None:
        sources = filtered_retrieve(question, filters=my_filter, top_k=top_k)
        method = "filtered"
    else:
        sources = naive_retrieve(question, top_k=top_k)
        method = "naive"

    # Step 3: Fallback if filtering returns empty
    if not sources and my_filter is not None:
        sources = naive_retrieve(question, top_k=top_k)
        method = "naive (fallback)"

    # Step 4: Build reasoning
    if my_filter is not None:
        filter_type = my_filter.get("doc_type", "?")
        if method == "naive (fallback)":
            reasoning = (
                f"Classified as '{filter_type}' but filter returned 0 results. "
                f"Fell back to naive retrieval -> {len(sources)} results."
            )
        else:
            reasoning = (
                f"Classified as '{filter_type}' -> filtered by doc_type='{filter_type}' "
                f"-> {len(sources)} results."
            )
    else:
        reasoning = f"No clear document type detected -> naive retrieval -> {len(sources)} results."

    # Step 5: Collect metadata
    source_types = [s["metadata"].get("doc_type", "?") for s in sources]
    top_score = sources[0]["score"] if sources else 0.0

    return {
        "question": question,
        "method": method,
        "filter_applied": my_filter,
        "sources": sources,
        "reasoning": reasoning,
        "top_score": top_score,
        "source_types": source_types,
    }

In [ ]:
# Test auto_classify_and_retrieve() on a diverse set of queries

test_queries = [
    "What is Northbrook's vacation policy?",
    "What were the Q3 revenue numbers?",
    "What decisions were made at the board meeting?",
    "What did the CEO say about company direction?",
    "What services does Northbrook offer?",
    "What is the expense reimbursement limit for hotels?",
    "How is the cloud migration progressing?",
    "When is the office relocation happening?",
]

print(f"{'#':>2}  {'Method':<18} {'Filter':>15} {'Score':>6} {'Types':<30} Question")
print("-" * 110)

all_results = []
for i, q in enumerate(test_queries, 1):
    result = auto_classify_and_retrieve(q, top_k=5)
    all_results.append(result)

    filter_str = str(result["filter_applied"].get("doc_type", "?")) if result["filter_applied"] else "none"
    types_str = str(result["source_types"][:3])
    print(f"{i:>2}  {result['method']:<18} {filter_str:>15} {result['top_score']:>6.3f} {types_str:<30} {q[:50]}")

# Summary
methods = [r["method"] for r in all_results]
filtered_count = sum(1 for m in methods if "filtered" in m)
naive_count = sum(1 for m in methods if m == "naive")
fallback_count = sum(1 for m in methods if "fallback" in m)

print(f"\nRouting summary: {filtered_count} filtered, {naive_count} naive, {fallback_count} fallback")

In [ ]:
# Detailed view: show the full reasoning for each query

for i, result in enumerate(all_results, 1):
    print(f"\n{'=' * 70}")
    print(f"Query {i}: {result['question']}")
    print(f"Method: {result['method']}")
    print(f"Filter: {result['filter_applied']}")
    print(f"Reasoning: {result['reasoning']}")
    print(f"Top score: {result['top_score']:.3f}")
    print(f"\nSources:")
    for s in result["sources"][:3]:
        src = s["metadata"].get("source", "?")
        dtype = s["metadata"].get("doc_type", "?")
        print(f"  [{s['score']:.3f}] {src:<40} type={dtype}")

### What to Check in Your Results

| Signal | Meaning |
|--------|---------|
| All "filtered" with correct types | Your `build_filter()` is working well |
| "naive (fallback)" appears | Filtering returned 0 results -- check if the filter was too restrictive |
| "naive" for ambiguous queries | Correct! Some questions do not have a clear document type |
| Mixed source_types in filtered results | Something is wrong -- filtered results should all match the filter |
| High scores on filtered results | Filter narrowed to the right pool and similarity did the rest |

**The key insight:** You just built a simple routing pipeline. The question goes in, your code decides HOW to retrieve, and the results come out with an explanation. This is the pattern behind every production RAG system -- classify the query, route to the right retrieval strategy, and log the reasoning.

---

## 8. PCA Visualization: The Embedding Gap

Before we close, there is something fundamental about **all** similarity-based retrieval -- naive and filtered alike -- that you need to see.

We project all corpus chunks into 2D using PCA (the same technique from Session 2.1), then add questions to see where they land.

**Connection to Session 2.1:** In Session 2.1, you saw that embedding models cluster similar phrases together -- finance phrases near finance, HR near HR. That is why semantic search works. Now you will see the **limitation**: questions and answers cluster in different neighborhoods.

**Connection to Session 3.1:** In Session 3.1, you saw that "WFH" missed remote_work_policy because of vocabulary mismatch. We will visualize that mismatch in embedding space and see exactly how far apart those phrasings land.

In [ ]:
# Load the pre-computed PCA transformer from the instructor demo
# This ensures students see the SAME coordinate system as the instructor's visualization

import pickle
from pathlib import Path
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca_path = Path("pca_transformer.pkl")

if pca_path.exists():
    with open(pca_path, "rb") as f:
        pca_data = pickle.load(f)

    pca = pca_data["pca"]
    chunk_pca = pca_data["chunk_coords_3d"]
    doc_types = pca_data["chunk_doc_types"]
    chunk_sources = pca_data["chunk_sources"]
    chunk_texts = pca_data["chunk_texts"]
    chunk_embeddings = pca_data["chunk_embeddings"]
    color_map = pca_data["doc_type_colors"]
    unique_types = pca_data["doc_type_names"]

    # For compatibility with Plotly hover text
    chunk_metadatas = [{"doc_type": dt, "source": src} for dt, src in zip(doc_types, chunk_sources)]

    print(f"Loaded PCA transformer from {pca_path}")
    print(f"  Chunks: {len(chunk_pca)}")
    print(f"  Document types: {unique_types}")
    print(f"  Variance explained: {pca.explained_variance_ratio_.sum():.1%}")
else:
    print(f"PCA pickle not found at {pca_path}")
    print("Falling back to computing from ChromaDB...")

    collection = get_collection()
    all_data = collection.get(
        include=["embeddings", "metadatas", "documents"],
        limit=collection.count()
    )

    chunk_embeddings = np.array(all_data["embeddings"])
    chunk_metadatas = all_data["metadatas"]
    chunk_texts = all_data["documents"]

    doc_types = [m.get("doc_type", "unknown") for m in chunk_metadatas]
    unique_types = sorted(set(doc_types))
    chunk_sources = [m.get("source", "unknown") for m in chunk_metadatas]

    pca = PCA(n_components=3)
    chunk_pca = pca.fit_transform(chunk_embeddings)

    color_map = {
        "policy": "#3b82f6",
        "meeting": "#22c55e",
        "financial": "#f97316",
        "memo": "#a855f7",
        "unknown": "#6b7280",
    }

    print(f"Computed PCA from {len(chunk_embeddings)} chunks")
    print(f"  Document types: {unique_types}")
    print(f"  Variance explained: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# Display PCA variance information
# The PCA transformer captures a fraction of the total variance --
# think of it like a photograph of a sculpture: you can see the shape,
# but you lose some detail from the angles you cannot see.

print(f"PCA: {chunk_embeddings.shape[1] if hasattr(chunk_embeddings, 'shape') else '?'} dims → 3 dims")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.1%}")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.1%}")
print(f"  PC3: {pca.explained_variance_ratio_[2]:.1%}")
print(f"\nUsing {'pre-computed' if pca_path.exists() else 'freshly computed'} PCA transformer.")
print("New questions will be projected into the same coordinate space as the instructor demo.")

In [ ]:
# Step 3: Create the base 2D scatter plot of corpus chunks by doc_type
# Using the first two PCA components

color_map = {
    "policy": "#3b82f6",     # blue
    "meeting": "#22c55e",    # green
    "financial": "#f97316",  # orange
    "memo": "#a855f7",      # purple
    "unknown": "#6b7280",   # gray
}

fig, ax = plt.subplots(figsize=(12, 8))

for doc_type in unique_types:
    mask = [dt == doc_type for dt in doc_types]
    indices = [i for i, m in enumerate(mask) if m]
    color = color_map.get(doc_type, "#6b7280")
    ax.scatter(
        chunk_pca[indices, 0],
        chunk_pca[indices, 1],
        c=color,
        label=doc_type,
        alpha=0.6,
        s=40,
        edgecolors="white",
        linewidth=0.5,
    )

ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_title("Northbrook Corpus Chunks in Embedding Space (PCA)")
ax.legend(title="Document Type")
plt.tight_layout()
plt.show()

print("See the clusters? Similar content lives in similar regions.")
print("This is why semantic search works at all.")

In [ ]:
# Step 4: Now embed some questions and see where they land
# This is the key insight: questions do NOT land on top of their answer chunks

demo_questions = [
    "What is Northbrook's vacation policy?",
    "What were the Q3 revenue numbers?",
    "What decisions were made at the board meeting?",
    "How does the performance review process work?",
    "What is the remote work policy?",
]

print("Embedding questions...")
question_embeddings = embed_texts(demo_questions)
question_pca = pca.transform(np.array(question_embeddings))

# Plot chunks + questions together
fig, ax = plt.subplots(figsize=(12, 8))

# Plot corpus chunks (smaller, transparent)
for doc_type in unique_types:
    mask = [dt == doc_type for dt in doc_types]
    indices = [i for i, m in enumerate(mask) if m]
    color = color_map.get(doc_type, "#6b7280")
    ax.scatter(
        chunk_pca[indices, 0],
        chunk_pca[indices, 1],
        c=color,
        label=f"chunks: {doc_type}",
        alpha=0.4,
        s=30,
        edgecolors="white",
        linewidth=0.5,
    )

# Plot questions as large red markers
ax.scatter(
    question_pca[:, 0],
    question_pca[:, 1],
    c="#ef4444",
    marker="X",
    s=200,
    label="QUESTIONS",
    edgecolors="black",
    linewidth=1.5,
    zorder=5,
)

# Label each question
for i, q in enumerate(demo_questions):
    # Truncate long labels
    label = q[:35] + "..." if len(q) > 35 else q
    ax.annotate(
        label,
        (question_pca[i, 0], question_pca[i, 1]),
        fontsize=7,
        ha="left",
        va="bottom",
        xytext=(5, 5),
        textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="#fef2f2", alpha=0.8),
    )

ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_title("The Embedding Gap: Questions vs Answer Chunks")
ax.legend(title="Type", loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

print("\nRed X markers = questions. Colored dots = corpus chunks.")
print("\nNotice: questions cluster TOGETHER, away from the answer chunks.")
print("The question 'What is the vacation policy?' is closer to OTHER questions")
print("than to the policy chunks that contain the actual answer.")

In [ ]:
# Step 5: Try 3D visualization with Plotly (if available)
# Falls back gracefully to 2D matplotlib if Plotly is not installed

try:
    import plotly.graph_objects as go

    # Build traces for each document type
    traces = []
    for doc_type in unique_types:
        mask = [dt == doc_type for dt in doc_types]
        indices = [i for i, m in enumerate(mask) if m]
        color = color_map.get(doc_type, "#6b7280")

        hover_texts = [
            f"Source: {chunk_metadatas[j].get('source', '?')}<br>"
            f"Type: {chunk_metadatas[j].get('doc_type', '?')}<br>"
            f"Chunk: {chunk_metadatas[j].get('chunk_index', '?')}<br>"
            f"Text: {chunk_texts[j][:80]}..."
            for j in indices
        ]

        traces.append(go.Scatter3d(
            x=chunk_pca[indices, 0],
            y=chunk_pca[indices, 1],
            z=chunk_pca[indices, 2],
            mode="markers",
            name=f"chunks: {doc_type}",
            marker=dict(size=4, color=color, opacity=0.6),
            hovertext=hover_texts,
            hoverinfo="text",
        ))

    # Add question markers
    question_hover = [f"QUESTION: {q}" for q in demo_questions]
    traces.append(go.Scatter3d(
        x=question_pca[:, 0],
        y=question_pca[:, 1],
        z=question_pca[:, 2],
        mode="markers+text",
        name="QUESTIONS",
        marker=dict(size=8, color="#ef4444", symbol="x", opacity=1.0),
        text=[q[:30] + "..." for q in demo_questions],
        textposition="top center",
        textfont=dict(size=8),
        hovertext=question_hover,
        hoverinfo="text",
    ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title="The Embedding Gap: Questions vs Answer Chunks (3D PCA)",
        scene=dict(
            xaxis_title="PC1",
            yaxis_title="PC2",
            zaxis_title="PC3",
        ),
        width=900,
        height=650,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    )
    fig.show()

    print("\nRotate the 3D plot to see the gap from different angles.")
    print("Hover over points to see which document/question they represent.")

except ImportError:
    print("Plotly not installed -- 2D matplotlib visualization shown above is sufficient.")
    print("Install with: pip install plotly   (optional, for interactive 3D exploration)")

### Understanding the Embedding Gap

What you are seeing:

- **Corpus chunks** cluster by document type. Policy chunks are near other policy chunks. Meeting notes cluster together.
- **Questions** cluster together, in a *different* region from the answer chunks.
- The question "What is the vacation policy?" is semantically closer to *other questions about vacation* than to the statement "Employees accrue 15 days of PTO per year."

**Why?** Questions have interrogative structure. Answers have factual statement structure. Different structures produce different embedding signatures.

**What this means for RAG:**
- Semantic similarity can bridge small gaps -- that is why RAG works at all
- But sometimes the gap is too wide, and the question is closer to a document that *mentions* the topic than to the one that *answers* it
- **Filtering does not fix this.** Filtering narrows the candidate pool, but within that pool, the gap is still there

**In AI-3, you will learn two techniques to close this gap:**
1. **HyDE** -- transform the query to sound like a hypothetical answer before searching
2. **Question Enrichment** -- add "what questions does this chunk answer?" to chunks at ingestion time

For now, just see the problem. Hold onto this insight.

In [ ]:
# YOUR TURN: Vocabulary Mismatch Experiment
# In Session 3.1, you saw that "WFH" missed remote_work_policy.
# Let's SEE why in embedding space.

# Step 1: Embed three phrasings of the same question
vocab_test = [
    "What is the remote work policy?",       # formal -- matches corpus vocabulary
    "WFH guidelines?",                        # abbreviation -- missed in 3.1
    "telecommute rules?",                     # synonym -- different vocabulary
]

print("Embedding 3 phrasings of the same intent...\n")
vocab_embeddings = embed_texts(vocab_test)
vocab_pca = pca.transform(np.array(vocab_embeddings))

# Step 2: Plot with corpus chunks
fig, ax = plt.subplots(figsize=(12, 8))

# Corpus chunks (faded)
for doc_type in unique_types:
    mask = [dt == doc_type for dt in doc_types]
    indices = [i for i, m in enumerate(mask) if m]
    color = color_map.get(doc_type, "#6b7280")
    ax.scatter(
        chunk_pca[indices, 0], chunk_pca[indices, 1],
        c=color, label=f"chunks: {doc_type}", alpha=0.3, s=25,
        edgecolors="white", linewidth=0.5,
    )

# Demo questions (smaller)
ax.scatter(
    question_pca[:, 0], question_pca[:, 1],
    c="#ef4444", marker="X", s=100, label="Demo questions",
    edgecolors="black", linewidth=1, alpha=0.4, zorder=5,
)

# Vocabulary variants (large, distinct colors)
vocab_colors = ["#22c55e", "#f59e0b", "#8b5cf6"]  # green, amber, purple
vocab_markers = ["*", "D", "P"]  # star, diamond, pentagon
for i, (q, color, marker) in enumerate(zip(vocab_test, vocab_colors, vocab_markers)):
    ax.scatter(
        vocab_pca[i, 0], vocab_pca[i, 1],
        c=color, marker=marker, s=350, label=q,
        edgecolors="black", linewidth=1.5, zorder=10,
    )

# Draw lines between the 3 phrasings to show spread
for i in range(len(vocab_test)):
    for j in range(i + 1, len(vocab_test)):
        ax.plot(
            [vocab_pca[i, 0], vocab_pca[j, 0]],
            [vocab_pca[i, 1], vocab_pca[j, 1]],
            "k--", alpha=0.3, linewidth=1,
        )

ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_title("Vocabulary Mismatch: Same Intent, Different Embeddings")
ax.legend(title="Type", loc="upper left", fontsize=7)
plt.tight_layout()
plt.show()

# Step 3: Measure the spread
print("\nSimilarity between phrasings (same intent, different words):")
for i in range(len(vocab_test)):
    for j in range(i + 1, len(vocab_test)):
        sim = cosine_similarity(vocab_embeddings[i], vocab_embeddings[j])
        print(f"  '{vocab_test[i]}' <-> '{vocab_test[j]}': {sim:.4f}")

# Step 4: Check what each phrasing actually retrieves
print("\nWhat does each phrasing retrieve?")
for q in vocab_test:
    results = naive_retrieve(q, top_k=3)
    sources = [r["metadata"].get("source", "?") for r in results]
    top_score = results[0]["score"] if results else 0
    print(f"  '{q}' -> [{top_score:.3f}] {sources[0]}")

print("\nSame intent. Different words. Different retrieval results.")
print("This is why vocabulary mismatch is the #1 retrieval failure.")

In [ ]:
# Measure the embedding gap AND show retrieval neighborhoods

print("=" * 85)
print("PART 1: Embedding Gap -- Question vs Answer Distance")
print("=" * 85)
print()
print(f"{'Question':<45}  {'Nearest ANY':>12}  {'Nearest Expected':>17}  {'Gap':>6}")
print("-" * 85)

expected_types = ["policy", "financial", "meeting", "policy", "policy"]

for q, q_emb, expected in zip(demo_questions, question_embeddings, expected_types):
    sims = [cosine_similarity(q_emb, chunk_emb.tolist()) for chunk_emb in chunk_embeddings]
    best_any = max(sims)
    expected_sims = [s for s, dt in zip(sims, doc_types) if dt == expected]
    best_expected = max(expected_sims) if expected_sims else 0.0
    gap = best_any - best_expected
    label = q[:43] + ".." if len(q) > 43 else q
    print(f"{label:<45}  {best_any:>12.4f}  {best_expected:>17.4f}  {gap:>+6.4f}")

print("\nPositive gap = nearest chunk overall is NOT of the expected type.")

# PART 2: Retrieval Neighborhoods -- What Each Pipeline Actually Pulls
# Using Q3 revenue query because it produces clear cross-type contamination in naive
print()
print("=" * 85)
print("PART 2: Retrieval Neighborhoods -- What Each Pipeline Actually Pulls")
print("=" * 85)

neighborhood_query = "What were the Q3 revenue numbers?"
naive_src = naive_retrieve(neighborhood_query, top_k=5)
filtered_src = filtered_retrieve(neighborhood_query, filters={"doc_type": "financial"}, top_k=5)

print(f"\nQuery: {neighborhood_query}")
print(f"\n  NAIVE (top 5):")
naive_types = {}
for r in naive_src:
    dt = r["metadata"].get("doc_type", "?")
    naive_types[dt] = naive_types.get(dt, 0) + 1
    print(f"    [{r['score']:.3f}] {r['metadata'].get('source', '?'):<35} type={dt}")
print(f"  Type distribution: {dict(naive_types)}")

print(f"\n  FILTERED (top 5, doc_type='financial'):")
filtered_types = {}
for r in filtered_src:
    dt = r["metadata"].get("doc_type", "?")
    filtered_types[dt] = filtered_types.get(dt, 0) + 1
    print(f"    [{r['score']:.3f}] {r['metadata'].get('source', '?'):<35} type={dt}")
print(f"  Type distribution: {dict(filtered_types)}")

print(f"\n  Naive pulls from {len(naive_types)} doc types. Filtered pulls from {len(filtered_types)}.")
print("  Filtering concentrates retrieval -- fewer types, more relevant chunks.")

---

## 9. The Blind Test (Optional)

Quick exercise: two answers to the same question -- Pipeline A and Pipeline B. You do not know which is naive and which is filtered. Read both. Decide which is better. Then reveal.

This removes confirmation bias. If you know which pipeline produced the answer, you might unconsciously favor the one you expect to be better.

In [ ]:
import random

def blind_test(query, filters, seed=None):
    """Run a blind comparison: show two answers without revealing which pipeline produced which."""
    if seed is not None:
        random.seed(seed)

    # Generate both answers
    naive_response = naive_rag(query, top_k=5)
    filtered_response = filtered_rag(query, filters=filters, top_k=5)

    # Randomly assign to A and B
    if random.random() > 0.5:
        answer_a, answer_b = naive_response, filtered_response
        reveal = {"A": "NAIVE", "B": "FILTERED"}
    else:
        answer_a, answer_b = filtered_response, naive_response
        reveal = {"A": "FILTERED", "B": "NAIVE"}

    print(f"Question: {query}")
    print()
    print("=" * 70)
    print("ANSWER A")
    print("=" * 70)
    print(answer_a.answer)
    print(f"\n  Sources: {len(answer_a.sources)} | Tokens: {answer_a.input_tokens}+{answer_a.output_tokens}")

    print()
    print("=" * 70)
    print("ANSWER B")
    print("=" * 70)
    print(answer_b.answer)
    print(f"\n  Sources: {len(answer_b.sources)} | Tokens: {answer_b.input_tokens}+{answer_b.output_tokens}")

    print()
    print("Which answer is better? A or B?")
    print("(Think about: focus, accuracy, source citation, completeness)")

    return reveal

In [ ]:
# Run the blind test
reveal = blind_test(
    "What is Northbrook Partners' vacation policy?",
    filters={"doc_type": "policy"},
    seed=42
)

In [ ]:
# REVEAL: Which pipeline produced which answer?
# Only run this cell AFTER you have decided which is better.

print("REVEAL:")
print(f"  Answer A was: {reveal['A']}")
print(f"  Answer B was: {reveal['B']}")
print()
print("Were you right? Did the better answer come from the pipeline you expected?")

---

## 10. Bridge to Observability

You built three functions today that form a complete retrieval system:
- `build_filter()` -- classifies question intent
- `score_comparison()` -- measures retrieval quality
- `auto_classify_and_retrieve()` -- wires everything together with reasoning

But consider: you ran 8 queries and read every result. What if you had 1,000 queries? How would you know if your pipeline started giving subtly wrong answers?

You need **visibility into your pipeline.** Logging, metrics, and tracing.

**Next session (4.1):** Observability and Debugging. You will build a `PipelineLogger` that captures query details, retrieval results, latency, token usage, and quality signals.

### Questions to Think About

> **On Visibility:** You built the pipeline -- how do you know it is working at scale?

> **On Debugging:** When `auto_classify_and_retrieve()` gives a bad answer, was it the classification? The filter? The retrieval? The generation?

> **On the Gap:** Filtering narrows the candidate pool. But the embedding gap remains. What else could you do? (Hint: what if the question sounded like an answer before you searched?)

---

## 11. Session Recap

### What You Built Today

1. **`build_filter()`** (YELLOW) -- analyzed question intent and returned ChromaDB `where` clauses
2. **`score_comparison()`** (YELLOW) -- computed precision, overlap, and verdict metrics for naive vs filtered
3. **`auto_classify_and_retrieve()`** (ORANGE) -- wired classification, retrieval, fallback, and reasoning into one pipeline

### What You Explored

4. **ChromaDB `where` clauses** -- one parameter that changes retrieval from "search everything" to "search within a category"
5. **The over-filtering trap** -- more filters is not always better; zero results is worse than noisy results
6. **PCA: the embedding gap** -- questions and answers cluster in different neighborhoods; filtering does not fix this
7. **Vocabulary mismatch in embedding space** -- "WFH", "remote work policy", and "telecommute rules" land in different spots despite the same intent

### Key Takeaways

- **Filters run BEFORE similarity search** -- ChromaDB narrows first, then ranks
- **Lower similarity + right document type > higher similarity + wrong document type**
- **Vocabulary mismatch is structural** -- the embedding model encodes phrasing differences as distance
- **The embedding gap is real** -- questions cluster with questions, not answers; AI-3 addresses this with HyDE and question enrichment
- **Every production RAG system routes queries** -- what you built today is the foundation

### Lab 2 -- Assigned Today

**Lab 2: RAG Pipeline Evaluation -- Naive vs Metadata-Aware**

| Field | Detail |
|-------|--------|
| **Due** | Start of Session 4.2 (two weeks) |
| **Weight** | 20% of course grade |
| **Foundation** | Today's functions: `build_filter()`, `score_comparison()`, `auto_classify_and_retrieve()` |

**What you submit:**
1. Both pipeline implementations -- naive and your auto-classify pipeline, working correctly
2. Test query set -- 10+ queries run against both pipelines
3. Comparison report -- for each query: scores, sources, precision metrics, and verdict
4. Analysis -- identify 3+ cases where filtering helped and 1+ case where it hurt; explain WHY
5. Metrics -- integrate PipelineLogger from Session 4.1

**Rubric:**

| Criterion | Points |
|-----------|--------|
| Pipeline Implementation | 25 |
| Test Coverage | 25 |
| Comparison Analysis | 25 |
| Metrics & Observability | 25 |

**Start from today's code.** Your `auto_classify_and_retrieve()` is the filtered pipeline. `naive_rag()` is the baseline. Run them head-to-head.

### Next Session: 4.1 -- Observability and Debugging

You built the pipeline. Now: how do you see inside it?